In [2]:
def volume_control(input, output, gain):
    for i in range(1024):
        output[i] = input[i] * gain

In [3]:
def echo_control(input, output, delay, decay):
    for i in range(1024):
        output[i] = input[i] + input[i - delay] * decay

In [4]:
import numpy as np
from scipy.io import wavfile

def pitch_shift(input, fs, tones, N=1024, H=256):
    ratio = 2 ** (tones/12)
    hann = 0.5 * (1 - np.cos(2*np.pi * np.arange(N)/ (N - 1)))
    output = np.zeros(len(input)+ N)

    for start in range (0, len(input) - N, H):
            frame = input [start : start + N] * hann
            spectrum = np.fft.fft(frame)
            shifted = np.zeros(N, dtype=complex)
            for k in range(N // 2):
                k_new = int(round(k * ratio))
                if 0 < k_new < N // 2: 
                    shifted[k_new] += spectrum[k]       
                    shifted[N - k_new] += spectrum[N - k]
            frame_out = np.real(np.fft.ifft(shifted))           
            output[start : start + N] += frame_out * hann  
    return output

In [5]:
def main():
    fs = 44100
    t = np.arange(fs) / fs
    audio = (np.sin(2 * np.pi * t) * 16000)
    num_samples = len(audio)

    #gain test
    gain = 1.5
    output = np.zeros(num_samples)
    num_blocks = num_samples // 1024
    for b in range(num_blocks):
        offset = b * 1024
        volume_control(audio[offset:offset+1024], output[offset:offset+1024], gain)
    rem = num_samples % 1024 # in case there are any leftover bytes remaining
    if rem > 0:
        offset = num_blocks * 1024
        for i in range(offset, offset+rem):
            output[i] = audio[i] * gain

    print(f"Gain Test (gain={gain})")
    print(f"input range: [{min(audio)}, {max(audio)}]")
    print(f"output range: [{min(output)}, {max(output)}]")
    
    #echo test

    #transcription test
    result = pitch_shift(audio, fs, tones=3)
    print("Input length:", len(audio))
    print("Output length:", len(result))
    print("First 5 output samples:", result[:5])

    

In [6]:
if __name__ == "__main__":
    main()

Gain Test (gain=1.5)
input range: [-16000.0, 16000.0]
output range: [-24000.0, 24000.0]
Input length: 44100
Output length: 45124
First 5 output samples: [ 0.         -0.00548402 -0.02192785 -0.04931959 -0.08764786]
